# 神经半衰期模型

[English](../../06_neural_hlr.ipynb) · 简体中文

两个模型分别包含与不包含 lexeme；所需 HLR 初始化权重会自动读取或训练。


In [1]:
# Locate shared data and modules when running from a subdirectory.
from pathlib import Path
import os
import sys

_notebook_start = Path.cwd().resolve()
_notebook_parents = (_notebook_start, *_notebook_start.parents)
_project_root = next(
    (p for p in _notebook_parents
     if (p / "data/raw/settles.acl16.learning_traces.13m.csv.gz").is_file()),
    None,
)
if _project_root is None:
    _project_root = next(
        (p for p in _notebook_parents if (p / "experiment_pipeline.py").is_file()),
        _notebook_start,
    )
os.chdir(_project_root)
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from experiment_pipeline import Pipeline
run = Pipeline(language="zh-CN")
globals().update(run.neural_hlr())


原始文件： D:\fujing123\Projects\Data-Analytic\HLR\data\raw\settles.acl16.learning_traces.13m.csv.gz
前 5 行：


,p_recall,timestamp,delta,user_id,learning_language,ui_language,lexeme_id,lexeme_string,history_seen,history_correct,session_seen,session_correct
0,1.0,1362076081,27649635,u:FO,de,en,76390c1350a8dac31186187e2fe1e178,lernt/lernen<vblex><pri><p3><sg>,6,4,2,2
1,0.5,1362076081,27649635,u:FO,de,en,7dfd7086f3671685e2cf1c1da72796d7,die/die<det><def><f><sg><nom>,4,4,2,1
2,1.0,1362076081,27649635,u:FO,de,en,35a54c25a2cda8127343f6a82e6f6b7d,mann/mann<n><m><sg><nom>,5,4,1,1
3,0.5,1362076081,27649635,u:FO,de,en,0cf63ffe3dda158bc3dbd55682b355ae,frau/frau<n><f><sg><nom>,6,5,2,1
4,1.0,1362076081,27649635,u:FO,de,en,84920990d78044db53c1b012f5bf9ab5,das/das<det><def><nt><sg><nom>,4,4,1,1


Reused saved duplicate check and cleaned dataset.


,raw_rows,exact_duplicates_removed,clean_rows
0,12854226,81,12854145


Model and EDA input: D:\fujing123\Projects\Data-Analytic\HLR\data\processed\settles.acl16.learning_traces.13m.dedup_v1.csv.gz


,split,start,stop_exclusive,rows
0,Train,0,9254984,9254984
1,Validation,9254984,11568730,2313746
2,Test,11568730,12854145,1285415


切分依据：原始 CSV 文件行顺序；随机打乱仅发生在 Train 内的训练阶段。
已复用与当前原始数据、切分一致的特征缓存。
缓存：D:\fujing123\Projects\Data-Analytic\HLR\data\interim\takeaway_hlr_features_v1.dat；539.4 MiB
基础特征： ['sqrt(1+history_correct)', 'sqrt(1+history_seen-history_correct)', 'bias']
Train 词项数：18,353；Validation 未见词项行数：1,150；Test 未见词项行数：560
Reused Train-only categorical index cache.
Train categories: 92,392 users, 6 learning languages, 4 UI languages.
Validation unseen users: 590,955
Loaded baselines results: D:\fujing123\Projects\Data-Analytic\HLR\data\artifacts\baselines-00e8f437f57573f2.joblib
Loaded nn results: D:\fujing123\Projects\Data-Analytic\HLR\data\artifacts\nn-e1311c401f47ef3a.joblib


## 7.1 N-HLR：包含 lexeme

使用完整 HLR 的权重初始化前两个隐藏单元；词项 ID 通过权重查表进入网络。epoch 0 应与完整 HLR 的 Validation MAE 一致。下面依次显示逐轮 Validation、训练资源和选定轮次的 Validation 指标。

In [2]:
from IPython.display import display
print("nn_hlr_trace")
display(nn_hlr_trace)
print("nn_hlr_validation")
display(nn_hlr_validation)
print("nn_hlr_resource")
display(nn_hlr_resource)


nn_hlr_trace


,epoch,n_validation,MAE(p),RMSE(p),MAE(h)_days
0,0,2313746,0.125906,0.290691,117.103823
1,1,2313746,0.118069,0.288238,113.434805
2,2,2313746,0.117678,0.288125,112.200501
3,3,2313746,0.117690,0.288041,111.148330
4,4,2313746,0.117607,0.287964,111.328741
5,5,2313746,0.117474,0.287888,111.929520


nn_hlr_validation


,model,n_train,n_validation,MAE(p),RMSE(p),MAE(h)_days
0,N-HLR,9254984,2313746,0.117474,0.287888,111.92952


nn_hlr_resource


,model,n_train,input_dim,hidden,epochs,selected_epoch,batch_size,fit_seconds,rss_before_GiB,peak_rss_GiB
0,N-HLR,9254984,18356,4,5,5,8192,81.78,0.303223,0.711937


## 7.2 N-HLR -lex：不包含 lexeme

架构、loss 与训练行数保持相同，输入只保留 `x_correct`、`x_wrong` 和常数项。前两个隐藏单元从 HLR -lex 初始化；epoch 0 应与该基线的 Validation MAE 一致。

In [3]:
from IPython.display import display
print("nn_hlr_lexless_trace")
display(nn_hlr_lexless_trace)
print("nn_hlr_lexless_validation")
display(nn_hlr_lexless_validation)
print("nn_hlr_lexless_resource")
display(nn_hlr_lexless_resource)


nn_hlr_lexless_trace


,epoch,n_validation,MAE(p),RMSE(p),MAE(h)_days
0,0,2313746,0.126217,0.290753,119.786430
1,1,2313746,0.118083,0.288248,115.746804
2,2,2313746,0.117656,0.288099,113.356707
3,3,2313746,0.117690,0.288023,112.395582
4,4,2313746,0.117608,0.287948,112.230476
5,5,2313746,0.117471,0.287850,112.533668


nn_hlr_lexless_validation


,model,n_train,n_validation,MAE(p),RMSE(p),MAE(h)_days
0,N-HLR -lex,9254984,2313746,0.117471,0.28785,112.533668


nn_hlr_lexless_resource


,model,n_train,input_dim,hidden,epochs,selected_epoch,batch_size,fit_seconds,rss_before_GiB,peak_rss_GiB
0,N-HLR -lex,9254984,3,4,5,5,8192,71.327734,0.716419,0.744392
